# Module 3, OPTIONAL: Initialize, Build, and Populate Vectors in AWS OpenSearch Serverless

For this class we are going to predominantly be working with in-memory vector databases.  While these are great for learning and quick demonstrations, it is helpful to consider more scalable options.  Unfortunately, these are beyond the scope of this workshop, but this notebook is included for reference should you be interested in trying something different.

In this notebook we are going to create a connection to AWS OpenSearch Serverless (AOSS or OSS, for short).  AOSS is an on-demand, serverless option for Amazon OpenSearch Service that eliminates the operational complexity of provisioning, configuring, and tuning OpenSearch clusters.  So in this activity we are going to create our vector database and index in AOSS and show you quickly how to query it.

## A note on these imports

We are going to use a few different packages here.  First, we are using `opensearchpy`, which provides wrapper methods for typical OpenSearch REST APIs.  Second, we are going to be using some helper functions built by AWS for creating the security policies for working with AOSS.

In [ ]:
!pip install opensearch-py

In [ ]:
import boto3
import json
import time
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from utility import create_bedrock_execution_role, create_oss_policy_attach_bedrock_execution_role, create_policies_in_oss, interactive_sleep

In [ ]:
session = boto3.session.Session()
region = session.region_name

In [ ]:
suffix = "demo"
bucket_name = "..."
vector_store_name = f"bedrock-sample-rag-{suffix}"
index_name = f"bedrock-sample-index-{suffix}"

## Establish the AOSS client

In [ ]:
aoss_client = session.client('opensearchserverless')

## Initializing Bedrock Execution Role

This code block creates an IAM execution role for Amazon Bedrock using the specified S3 bucket, and then extracts the role's ARN from the returned response for later use in AWS operations.

In [ ]:
bedrock_kb_execution_role = create_bedrock_execution_role(bucket_name=bucket_name)
bedrock_kb_execution_role_arn = bedrock_kb_execution_role['Role']['Arn']

## Establishing and Verifying the AOSS Collection

This code block first creates the necessary security policies—encryption, network, and access—by calling the `create_policies_in_oss` helper function. It then attempts to create a new vector search collection using the specified vector store name.  If a collection with that name already exists, it catches the conflict exception and retrieves the existing collection details instead.  The code then extracts the collection ID from the response, checking for different response structures, and constructs the host URL based on that ID and the AWS region.  Finally, it enters a loop to poll the collection status every 15 seconds until the collection becomes active, confirming that the setup is complete before proceeding.

In [ ]:
encryption_policy, network_policy, access_policy = create_policies_in_oss(vector_store_name=vector_store_name,
                       aoss_client=aoss_client,
                       bedrock_kb_execution_role_arn=bedrock_kb_execution_role_arn)

try:
    collection_response = aoss_client.create_collection(name=vector_store_name, type='VECTORSEARCH')
    print("Collection created.")
except aoss_client.exceptions.ConflictException:
    print(f"Collection {vector_store_name} already exists. Retrieving existing collection details...")
    collection_response = aoss_client.batch_get_collection(names=[vector_store_name])

print("Creating collection...")

if 'createCollectionDetail' in collection_response:
    collection_id = collection_response['createCollectionDetail']['id']
elif 'collectionDetails' in collection_response:
    collection_id = collection_response['collectionDetails'][0]['id']
else:
    raise ValueError("Unable to determine collection ID from response")

host = f"{collection_id}.{region}.aoss.amazonaws.com"
print("Using host:", host)

while True:
    response = aoss_client.batch_get_collection(names=[vector_store_name])
    status = response['collectionDetails'][0]['status']
    if status == 'ACTIVE':
        break
    print("Collection is still being created. Waiting 15 seconds...")
    time.sleep(15)
    
print("Collection is active.")

## Configuring AWS Signature Authentication

This code retrieves AWS credentials from the active session and uses them to create a SigV4 authenticator for the Amazon OpenSearch Serverless (AOSS) service. This is necessary to securely sign and authenticate requests to AOSS, ensuring that all interactions with the service are authorized and secure.

In [ ]:
credentials = session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, region, service='aoss')

## Setting Up the OpenSearch Client

This code instantiates an OpenSearch client configured to connect securely to your AOSS host on port 443 using AWS SigV4 authentication, ensuring that all communications are encrypted (via SSL) and properly verified. It also sets a 300-second timeout, accommodating longer operations if needed.

In [ ]:
oss_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

## Creating the Vector Index

A vector index is a specialized data structure designed to store high-dimensional vector representations—such as embeddings generated from text—and supports efficient similarity searches (like nearest-neighbor queries). Creating a vector index is crucial for quickly retrieving relevant documents or data based on their semantic similarity, which is essential in applications like retrieval-augmented generation where you want to ground AI responses in up-to-date or domain-specific information.

This code configures an OpenSearch index for efficient k-nearest neighbor (kNN) vector search.  Several of these settings are beyond the scope of this course.  However, it is important to know that many LLMs use a vector dimension of 1536 and kNN search.  In the settings below, we enable kNN search ("index.knn": "true"), sets the index to use one shard and no replicas (ideal for smaller datasets or testing), and specify algorithm parameters such as "knn.algo_param.ef_search": 512 to control the trade-off between search accuracy and speed. 

In [ ]:
index_body = {
    "settings": {
        "index.knn": "true",
        "number_of_shards": 1,
        "knn.algo_param.ef_search": 512,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "vector": {
                "type": "knn_vector",
                "dimension": 1536,
                "method": {
                    "name": "hnsw",
                    "engine": "faiss",
                    "space_type": "l2"
                }
            },
            "text": { "type": "text" }
        }
    }
}

## Initializing the Vector Index in AOSS

This code block attempts to create an OpenSearch index using the defined name and configuration, and then waits 60 seconds to allow the index to initialize. If the index creation succeeds, it prints a confirmation message; otherwise, it catches and prints any errors that occur during the process.

In [ ]:
try:
    oss_client.indices.create(index=index_name, body=json.dumps(index_body))
    print("Index created. Waiting for it to initialize...")
    time.sleep(60)  # Wait for index creation
except Exception as e:
    print("Error creating index:", e)

## Generating the Embeddings

We now create a helper function that will use the Amazon Titan text embedding model to create our vectors.  Text is passed into this function as a chunk, which then returns the 1536-dimension vector.

In [ ]:
def generate_embedding(text):
    """
    Call an Amazon Bedrock embedding model to generate an embedding for the provided text.
    Replace the model_id and adjust the payload/response extraction based on your actual model.
    """
    # Create a boto3 client for Bedrock (ensure your environment has the required permissions)
    bedrock_client = boto3.client('bedrock-runtime')
    
    # Specify the model ID for your embedding model (update with your actual model ID)
    model_id = "amazon.titan-embed-text-v1"  # e.g., "amazon-bedrock-embedding-model"
    
    # Prepare the request payload; adjust keys according to your model's API requirements
    payload = {
        "inputText": text
    }
    
    # Invoke the model (the parameters may differ based on your actual API)
    response = bedrock_client.invoke_model(
        modelId=model_id,
        contentType='application/json',
        body=json.dumps(payload)
    )
    
    # Parse the response; here we assume the embedding is returned under the 'embedding' key
    result = json.loads(response['body'].read().decode('utf-8'))
    embedding = result.get('embedding')
    
    return embedding

Before we write it anywhere, let's see what that vector looks like:

In [ ]:
test_str = "Hello, World!"
embedding_vector = generate_embedding(test_str)
print('Vector length: ', len(embedding_vector))
print(embedding_vector[0:10])

##  Writing the Vectors to AOSS

Now that we know what this vector looks like, we want to write it to our database.  We do this by passing it in as a dictionary to our `oss_client`.

In [ ]:
# Create a document that includes the text and its embedding vector
document = {
    "text": s3_data,          # your original text
    "vector": embedding_vector  # the embedding vector you just generated
}

In [ ]:
response = oss_client.index(index=index_name, body=json.dumps(document))

print("Indexing response:")
print(json.dumps(response, indent=2))

Let's now see everything that is presently in the database.  We haven't added too much yet, so this won't be a problem.  However, this would obviously be impractical if we had many vectors stored.

In [ ]:
# Execute a match_all query to retrieve all documents in the index
search_query = {
    "query": {
        "match_all": {}
    }
}

search_response = oss_client.search(index=index_name, body=json.dumps(search_query))
print("Search Response:")
print(json.dumps(search_response, indent=2))

## Searching by Keyword

Now let's say we wanted to search the database by a specific keyword...

In [ ]:
keyword = "World"

query = {
    "query": {
        "match": {
            "text": keyword
        }
    }
}

# Execute the search query against your index
search_response = oss_client.search(index=index_name, body=json.dumps(query))

# Process the search response to print the text and its embedding vector
for hit in search_response["hits"]["hits"]:
    source = hit["_source"]
    print("Document found:")
    print("Text:", source["text"])
    print("Embedding:", source["vector"])

## Concluding Thoughts

In this activity we have seen how to create a vector database and its index on AOSS and write some vectors to it.  This is a great start!  Next we will learn how to add a few more vectors in a less manual way and use them in a pipeline to enhance our GenAI applications through the method of retrieval-augmented generation (RAG).